# 🏨 Hotel Reservation System 10: A Web-Search Agent

Guests ask about more than rooms: "What can I visit near the hotel?", "Which shops nearby are open now?". The
answers are not in the hotel's database and change over time, so the last agent in the build searches the web
with **Tavily**. The coordinator forwards such questions to it, exactly as it forwards bookings to the
reservation agent.

## Learning Objectives
1. **A search tool** - `TavilySearch` returns titles, URLs and snippets the model can cite
2. **A web-search agent** - A tool loop (`web_search_assistant ⇄ search_tools`) with its own grounded prompt
3. **One more route** - The coordinator names `web_search_assistant` for questions about the outside world
4. **Checking routes in code** - Which node ran, from `stream_mode="updates"`, for a POI and a booking question

## Where this fits
`09_hotel_system_wide_error_handling.ipynb` added recovery from failures. This notebook adds web search on top
of the core system of `03_hotel_compliance_checker.ipynb`. The language and sentiment nodes of
`08_hotel_guest_language_and_sentiment.ipynb` would sit in front of the coordinator in the full system; they are
left out here to keep the focus on search. This is the last hotel notebook;
`11_developer_vs_tester_agents.ipynb` starts a different multi-agent system.

## Prerequisites
- The core system from `03_hotel_compliance_checker.ipynb` (read it first; this notebook imports it from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `TAVILY_API_KEY` in the project-root `.env` (a free key is available at tavily.com)

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and `TAVILY_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import json
import os
from typing import Literal

from langchain_core.messages import SystemMessage
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all four agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: Data and Tools, Now With Web Search

The database, SQL tools and compliance retriever are notebook 03's. `TavilySearch` reads `TAVILY_API_KEY` from
the environment; `max_results=3` keeps each search small and cheap.

In [ ]:
# ============================================================================
# DATABASE, SQL TOOLS, COMPLIANCE RETRIEVER AND THE SEARCH TOOL
# ============================================================================
assert os.getenv("TAVILY_API_KEY"), "Add TAVILY_API_KEY to the project-root .env (see Prerequisites)"

hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools("hotel.db")  # 02's tools: run_query only reads, book_room is the only write
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]
search_tool = TavilySearch(max_results=3, topic="general")

print(f"✅ {len(sql_tools)} SQL tools, 1 retrieval tool, and the search tool '{search_tool.name}'")

### What the search tool returns
One direct call, before any agent is involved.

In [ ]:
# ============================================================================
# ONE DIRECT SEARCH
# ============================================================================
raw = search_tool.invoke({"query": f"top points of interest near {hc.HOTEL_ADDRESS}"})
for r in raw["results"]:
    print(f"- {r['title']}\n  {r['url']}\n  {r['content'][:150]}...\n")

**Reading the output:** three results, each with a title, a URL and a text snippet. The agent sees the same
fields, so it can answer from the snippets and cite the URLs. The results change over time; that is the point
of searching instead of relying on the model's training data.

---
## 🤖 Part 2: The Agents

Notebook 03's coordinator guideline, compliance checker and reservation agent (from `hotel_common`), plus:
- **`web_search_assistant`**: the search tool bound to the model, with a prompt that says where the hotel is and
  requires answers from search results only.
- **Coordinator guideline**: questions about places outside the hotel go to `web_search_assistant`.

In [ ]:
# ============================================================================
# COORDINATOR, COMPLIANCE, RESERVATION AND WEB-SEARCH AGENTS
# ============================================================================
AGENTS = ["reservation_assistant", "compliance_checker", "web_search_assistant"]
conversation_prompt = hc.conversation_prompt(
    AGENTS,
    extra_guidelines=hc.COMPLIANCE_FIRST + "\n"  # 03's guideline, plus:
    "- Questions about places outside the hotel (points of interest, restaurants, shops, opening hours, events "
    "nearby) need up-to-date information from the internet: forward them to web_search_assistant.\n"
    "- Mention an agent's name only when you forward to it.",
)

WEB_SEARCH_PROMPT = SystemMessage(content=f"""You are the hotel's web search agent. The hotel is at {hc.HOTEL_ADDRESS}.
- Always call the search tool before answering; never answer from your own knowledge.
- Search near the hotel's address unless the guest names another place.
- Answer the guest's question briefly from the search results only, and list the URLs you used.""")

sql_assistant, rag_assistant, generate = hc.make_core_nodes(llm, sql_tools, rag_tools)  # 03's agents, 04's verdict
llm_with_search_tools = llm.bind_tools([search_tool])


def conv_assistant(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + state["messages"])]}


def web_search_assistant(state: MessagesState) -> dict:
    return {"messages": [llm_with_search_tools.invoke([WEB_SEARCH_PROMPT] + state["messages"])]}


print("✅ Four agents defined")

---
## 🔀 Part 3: Router and Graph

`choose_next_node` is `hc.route_by_mention()`, 03's router and loop guard, with the new agent in the list. Once
the latest request has a compliance verdict, `compliance_checker` is dropped from the candidates, so a reply like
"the compliance checker approved it, forwarding to web_search_assistant" still reaches the search agent instead
of stopping.

In [ ]:
# ============================================================================
# ROUTER AND GRAPH
# ============================================================================
def choose_next_node(state: MessagesState) -> Literal["reservation_assistant", "compliance_checker", "web_search_assistant", "__end__"]:
    return hc.route_by_mention(state["messages"], AGENTS) or END  # 03's router, with its loop guard


builder = StateGraph(MessagesState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("compliance_checker", rag_assistant)
builder.add_node("web_search_assistant", web_search_assistant)
builder.add_node("generate", generate)
builder.add_node("sql_tools", ToolNode(sql_tools))
builder.add_node("rag_tools", ToolNode(rag_tools))
builder.add_node("search_tools", ToolNode([search_tool]))

builder.add_edge(START, "conv_assistant")
builder.add_conditional_edges(
    "conv_assistant", choose_next_node, ["reservation_assistant", "compliance_checker", "web_search_assistant", END]
)
builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: END})
builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
builder.add_conditional_edges("web_search_assistant", tools_condition, {"tools": "search_tools", END: END})
builder.add_edge("sql_tools", "reservation_assistant")
builder.add_edge("rag_tools", "generate")
builder.add_edge("search_tools", "web_search_assistant")
builder.add_edge("generate", "conv_assistant")

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## ▶️ Part 4: A Neighbourhood Question and a Booking

Two guests on two threads. `show_turn()` records the nodes that ran (with `hc.run_traced()`) so the next
cell can check the routes in code:
- a points-of-interest question should reach `web_search_assistant` **and** make it call the search tool
- a booking request should reach the reservation agent and **not** the web-search agent

In [ ]:
# ============================================================================
# TWO SCRIPTED TURNS WITH TRACED PATHS
# ============================================================================
def show_turn(text: str, thread_id: str) -> list[str]:
    """Run one guest message and return the names of the nodes that ran, in order."""
    nodes, state = hc.run_traced(graph, text, thread_id, verbose=False)
    print(f"👤 Guest: {text}")
    print(f"   path: {' → '.join(nodes)}")
    print(f"🤖 {state['messages'][-1].content}\n")
    return nodes


stay = date.today() + timedelta(days=60)
poi_question = "What are the top points of interest within walking distance of the hotel?"
booking_request = f"I'm Dana Lee and I'd like to book a Deluxe room from {stay} to {stay + timedelta(days=2)}."

poi_nodes = show_turn(poi_question, thread_id="web-poi")
booking_nodes = show_turn(booking_request, thread_id="web-booking")

### Did each question take the right route?

In [ ]:
# ============================================================================
# CHECK THE ROUTES AND LIST THE SOURCES
# ============================================================================
checks = [
    ("POI question reached web_search_assistant", "web_search_assistant" in poi_nodes),
    ("POI question called the search tool", "search_tools" in poi_nodes),
    ("Booking request did NOT reach web_search_assistant", "web_search_assistant" not in booking_nodes),
    ("Booking request reached reservation_assistant", "reservation_assistant" in booking_nodes),
]
for label, ok in checks:
    print(f"{'✅' if ok else '❌'} {label}")

print("\nSources the search agent received for the POI question:")
poi_messages = graph.get_state({"configurable": {"thread_id": "web-poi"}}).values["messages"]
for m in poi_messages:
    if m.type == "tool" and m.name == search_tool.name:
        try:
            for r in json.loads(m.content).get("results", []):
                print(f"  - {r['title']}: {r['url']}")
        except (json.JSONDecodeError, AttributeError):
            print("  (non-JSON tool output) " + str(m.content)[:200])

**Reading the output:** the first two ✅ lines show the POI question went coordinator → compliance check →
coordinator → `web_search_assistant` → `search_tools`, so the answer is built from live search results; compare
its URLs with the source list. The last two show the booking request was routed to the reservation agent and
never touched web search. A ❌ on the fourth line alone usually means the coordinator asked a follow-up question
first (read its reply); a ❌ on the third would mean a routing mistake.

To chat with the full system yourself, run `hc.chat_with_agent(graph, "my-thread")` in a new cell.

---
## 📝 Summary

- **`TavilySearch(max_results=3)`** gives an agent current, citable web results.
- The **web-search agent** is one more tool loop, with its own prompt: search first, answer from results, cite.
- The coordinator routes to it by name, like the other agents; the router drops `compliance_checker` once a
  request is checked, so mentioning the checker does not block the forward.
- `stream_mode="updates"` shows exactly which nodes ran, so routes are **checked in code**.

Notebooks 04–10 each added capabilities to the same core from 03 (coordinator, compliance checker and
reservation agent): edge-case guards, tests, parallel agents, translation and sentiment, error recovery and web
search. Each notebook adds its capability to that core on its own; no notebook composes them all into one graph.
Composing them is mostly wiring: `08_hotel_guest_language_and_sentiment.ipynb` composes two, running
`sentiment_analysis` right after `translate_in`, on the English text.

### Next Steps
- `11_developer_vs_tester_agents.ipynb`: a different multi-agent system, where a developer agent and a tester
  agent work on code together